# Learnings 03: Errors, Challenges, And Resolutions

This notebook captures the difficult parts of the journey: the failures we encountered, what they meant, and how we resolved them.

This notebook is important because real projects are not just about final working code. They are also about debugging skill, persistence, and understanding why a system failed.

## 1. The First Major Parsing Failure: Coroutine Passed Instead Of Parsed Content

### What happened
The ingestion pipeline failed with an error saying `pdf_content` received a coroutine object instead of parsed content.

### What it meant
An async function had been called without awaiting it, so instead of receiving the result, the system received the coroutine object itself.

### Why it mattered
This blocked all PDF parsing and caused Pydantic validation failures.

### Resolution
We corrected the async/sync boundary so the parser result was properly awaited or wrapped correctly.

## 2. Async And Sync Contract Mismatch In The Parser Layer

### What happened
One layer treated `parse_pdf()` as async, but the underlying Docling parser implementation was synchronous.

### Why this was tricky
These kinds of bugs are subtle because the method names can look correct while the execution model is inconsistent.

### Resolution
We kept the higher-level parser API async and executed the blocking Docling work in a thread so it fit the async pipeline cleanly.

## 3. Missing System Library For Docling

### What happened
Docling failed in the Airflow container because a required shared system library such as `libGL.so.1` was missing.

### Meaning
The Python package was installed, but the operating-system-level dependency was not.

### Resolution
We updated the Airflow Dockerfile to install the required runtime libraries. This moved the problem from "parser cannot start" to "parser can run."

## 4. Large PDFs Were Being Skipped Completely

### What happened
Long PDFs were skipped when they crossed the configured page limit.

### Why this was unsatisfying
Some papers were useful even if long, and often the important content was in the first pages while later pages contained references.

### Resolution
Instead of skipping the entire file, we changed behavior to process only the first configured pages. This improved coverage while controlling cost and latency.

## 5. Corrupted Or Invalid Cached PDFs

### What happened
Some cached PDFs were invalid or corrupted from the parser's point of view.

### Why it mattered
A broken cache can keep poisoning later runs if the system always trusts the local file.

### Resolution
We improved download handling and added logic to force a fresh redownload when a cached file looked corrupted.

## 6. Need For A Parser Fallback

### What happened
Docling still could not reliably parse every paper.

### Insight
In real pipelines, one parser is often not enough for all documents.

### Resolution
We added a fallback path using another PDF extraction method so the system could still rescue text for difficult documents.

## 7. Jina Embedding Rate Limits (`429` Errors)

### What happened
Indexing started failing with `429 Too Many Requests` from Jina.

### Meaning
We were sending too many embedding requests too quickly.

### Resolution
We added retry logic, backoff, smaller batches, and slower pacing. After that, indexing completed successfully.

## 8. OpenSearch Disk Watermark And Read-Only Block

### What happened
OpenSearch refused writes because disk usage crossed its flood-stage watermark, causing the index to become read-only.

### Why this mattered
Indexing could not proceed, even though the Python code itself was correct.

### Resolution
We freed disk space, cleared the read-only block, and restored indexing. This taught us that not every failure is a code failure; some are infrastructure failures.

## 9. Host Disk Exhaustion

### What happened
At one point the root filesystem became full. OpenSearch and Postgres started failing because they could not write temporary or checkpoint files.

### Symptoms
- `No space left on device`
- OpenSearch restart loops
- Postgres panic and recovery loops
- even local debugging tools had trouble using `/tmp`

### Resolution
We cleaned Docker caches and volumes and learned to distinguish project-level problems from host-level resource problems.

## 10. API Container Using Old Code

### What happened
Sometimes the source code looked fixed locally, but the running API still behaved like the old version.

### Meaning
The container had not been rebuilt from the latest source.

### Resolution
We rebuilt the API image and redeployed the container. This reinforced an important lesson: changing source files is not enough when Docker images are involved.

## 11. `/ask` Failing Because Of Tracing Wrapper Bugs

### What happened
The normal `/ask` endpoint was failing because the tracing wrapper was calling a method that did not exist on the Langfuse tracer object.

### Resolution
We updated the tracing wrapper to use the actual v3-style span methods. This removed the internal server error and restored normal RAG.

## 12. `/ask-agentic` Dependency Wiring Mismatch

### What happened
The agentic endpoint failed because a dependency factory received a `model` argument it did not accept.

### Resolution
We aligned the dependency and factory signatures and also passed the request model through to the agentic service.

## 13. Empty `sources` And Wrong `chunks_used` In Agentic Responses

### What happened
The agentic answer worked, but source URLs were empty and `chunks_used` was inaccurate.

### Why this happened
Relevant sources were not actually being propagated from tool messages into the final response, and the chunk count logic did not understand the tool message format.

### Resolution
We added source extraction, normalized arXiv URLs, and counted retrieved documents correctly from the graph output.

## 14. Telegram Bot Conflict With Multi-Worker API Startup

### What happened
The Telegram token worked, but polling failed with a conflict error.

### Root cause
The API container was running multiple workers, and each worker tried to start the same bot. Telegram allows only one polling consumer.

### Resolution
We moved the Telegram bot to a dedicated single-process service and stopped auto-starting it inside every API worker.

## 15. Langfuse Configuration And Trace ID Completion

### What happened
At first, Langfuse was present in Compose but not actually active in the app. Later, tracing worked but `trace_id` still came back as `null`.

### Why this happened
- wrong environment variable style at first
- then code did not include `trace_id` in the response
- some agentic nodes still used older span helper names

### Resolution
We fixed env variable naming, verified Langfuse initialization, added compatibility helpers, and returned `trace_id` from the agentic response.

## 16. Main Lessons From The Journey

The biggest lessons were:

1. Not all failures are code failures; many are environment or resource problems.
2. Async/sync mismatches can break pipelines in subtle ways.
3. Docker rebuilds matter; local source and running container can diverge.
4. Parsing pipelines need fallbacks and cache hygiene.
5. Retrieval quality and answer quality are different problems.
6. Deployment of background services must respect process model constraints.
7. Good observability becomes essential once the main system starts working.

This notebook is the memory of the hard-earned project knowledge that made the final system stable.